#### Ryanair Price Analysis: 
#### Scraping and Analyzing Ryanair Flight Prices
#### 2nd Part 
Merlin Mirley Moreno Palacios 
Date: 04/2026

#### 5. Loading the dataset

After completing the scraping process, the flight data is stored in a CSV file. The next step is to load this dataset into Python and inspect its structure before performing any cleaning or analysis.

This step is essential to verify that the data has been correctly collected and to identify potential issues such as missing values, incorrect data types, or inconsistencies.

In [55]:
import pandas as pd

df = pd.read_csv("flight_data.csv") # Replace "flight_data.csv" with the actual path to your CSV file
# check the first few rows of the dataframe to see the structure of the data
df.head(2)

,Scrape Date,Flight Number,Origin City,Origin Airport,Destination City,Destination Airport,Departure Date,Departure Time,Arrival Time,Price
0,"2026-02-18,FR 2663,Fráncfort Hahn,HHN,Alicante...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"*""",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


When loading the dataset, the data was not correctly split into columns. Instead, each row was interpreted as a single text string.

This problem is caused by formatting issues in the CSV file, particularly the use of quotation marks and commas within the data.

#### 5. Repairing the raw dataset structure

When the raw file was loaded, it became clear that the observations were not being parsed as a structured table. Instead, many rows were stored as quoted text strings, and the price variable contained embedded commas that disrupted the column separation.

To solve this issue, I read the file line by line, preserved the header, removed malformed quotation marks, filtered out corrupted rows, and manually reconstructed the price field. This made it possible to rebuild the dataset into a valid tabular format before proceeding with the cleaning and analysis stages.

In [56]:
import pandas as pd

# Read the raw file line by line
with open("flight_data.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()

# Extract the header from the first line
header = lines[0].strip().split(",")

# Initialize a list to store cleaned rows
rows = []

# Process each data row
for line in lines[1:]:
    line = line.strip()
    
    # Skip empty or malformed lines
    if not line or "*""" in line:
        continue
    
    # Remove outer quotation marks if present
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    
    # Remove duplicated quotation marks
    line = line.replace('""', '')
    
    # Split the row by commas
    parts = line.split(",")
    
    # Skip rows that do not contain enough elements
    if len(parts) < 9:
        continue
    
    # Keep the first 9 fixed columns
    fixed = parts[:9]
    
    # Reconstruct the price field, which may contain a decimal comma
    price = ",".join(parts[9:])
    
    # Rebuild the full row
    row = fixed + [price]
    
    # Append the cleaned row to the list
    rows.append(row)

# Create the DataFrame from the cleaned rows
df = pd.DataFrame(rows, columns=header)

# Remove unnamed or invalid columns if any were created
df = df.loc[:, df.columns.notna()]

# Display the first rows of the repaired dataset
df.head(2)


,Scrape Date,Flight Number,Origin City,Origin Airport,Destination City,Destination Airport,Departure Date,Departure Time,Arrival Time,Price
0,2026-02-18,FR 2663,Fráncfort Hahn,HHN,Alicante,ALC,2026-02-20,06:20,09:00,"156,99 €"
1,2026-02-18,FR 2663,Fráncfort Hahn,HHN,Alicante,ALC,2026-02-21,17:45,20:25,"60,99 €"


In [57]:
df.shape

(9258, 10)

In [58]:
df.columns

Index(['Scrape Date', 'Flight Number', 'Origin City', 'Origin Airport',
       'Destination City', 'Destination Airport', 'Departure Date',
       'Departure Time', 'Arrival Time', 'Price'],
      dtype='str')

In [59]:
df.dtypes

Scrape Date            str
Flight Number          str
Origin City            str
Origin Airport         str
Destination City       str
Destination Airport    str
Departure Date         str
Departure Time         str
Arrival Time           str
Price                  str
dtype: object

## 6. Dataset overview

The repaired dataset consists of 9,258 observations and 10 variables, corresponding to a shape of `(9258, 10)`.

The variables included are: `Scrape Date`, `Flight Number`, `Origin City`, `Origin Airport`, `Destination City`, `Destination Airport`, `Departure Date`, `Departure Time`, `Arrival Time`, and `Price`.

A preliminary inspection shows that all variables are currently stored as strings. As a result, the dataset requires additional type conversion and cleaning before it can be used for meaningful analysis.

In [60]:
df[(df["Price"] == "sold out")]

,Scrape Date,Flight Number,Origin City,Origin Airport,Destination City,Destination Airport,Departure Date,Departure Time,Arrival Time,Price
1822,2026-02-19 19:15:08,FR 2813,Colonia,CGN,Londres Stansted,STN,2026-03-29,11:45,12:05,sold out
3337,2026-02-20 15:41:59,FR 2813,Colonia,CGN,Londres Stansted,STN,2026-03-29,11:45,12:05,sold out
4357,2026-02-21 09:11:19,FR 2813,Colonia,CGN,Londres Stansted,STN,2026-03-29,11:45,12:05,sold out
5975,2026-02-22 10:59:18,FR 2813,Colonia,CGN,Londres Stansted,STN,2026-03-29,11:45,12:05,sold out
6443,2026-02-23 10:16:20,FR 2813,Colonia,CGN,Londres Stansted,STN,2026-03-29,11:45,12:05,sold out
8602,2026-02-24 08:46:40,FR 2813,Colonia,CGN,Londres Stansted,STN,2026-03-29,11:45,12:05,sold out


#### Handling sold-out observations

Some flights in the dataset are marked as "sold out", meaning that no price is available for those observations.

Since these entries do not contain valid price information, they cannot be used in numerical analysis. Therefore, I remove these observations from the dataset to ensure that all remaining data points correspond to actual prices.

In [63]:
# Replace "sold out" with missing values
df["Price"] = df["Price"].replace("sold out", pd.NA)

# Replace empty strings and whitespace with missing values
df["Price"] = df["Price"].replace(r"^\s*$", pd.NA, regex=True)

# Drop rows with missing prices
df = df.dropna(subset=["Price"])

# Reset index after dropping rows
df = df.reset_index(drop=True)

# Check new shape
df.shape

(9114, 10)

In [64]:
df["Price"].isna().sum()

np.int64(0)

In [65]:
# Create a copy of the original price column
df["Price_raw"] = df["Price"]

# Identify currency
df["currency"] = df["Price_raw"].str.extract(r"(€|£|MAD)")

# Clean numeric part of the price
df["Price_clean"] = (
    df["Price_raw"]
    .str.replace("€", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.replace("MAD", "", regex=False)
       # Remove thousand separators (dot)
    .str.replace(".", "", regex=False)
    
    # Replace decimal comma with dot
    .str.replace(",", ".", regex=False)
    
    # Remove spaces
    .str.strip()
)

# Convert to numeric
df["Price_clean"] = pd.to_numeric(df["Price_clean"], errors="coerce")

# Convert all prices to euros
df["Price_eur"] = df["Price_clean"]

# Apply conversion rates
df.loc[df["currency"] == "£", "Price_eur"] = df["Price_clean"] * 1.15
df.loc[df["currency"] == "MAD", "Price_eur"] = df["Price_clean"] * 0.092

# Final check
df[["Price_raw", "currency", "Price_clean", "Price_eur"]].head()

,Price_raw,currency,Price_clean,Price_eur
0,"156,99 €",€,156.99,156.99
1,"60,99 €",€,60.99,60.99
2,"110,99 €",€,110.99,110.99
3,"40,99 €",€,40.99,40.99
4,"63,99 €",€,63.99,63.99


In [49]:
df["currency"].value_counts()

currency
€      7944
£      1038
MAD     132
Name: count, dtype: int64

In [66]:
df["Price_eur"].describe()

count    9114.000000
mean       69.197874
std        43.491926
min        14.990000
25%        34.390000
50%        60.900000
75%        93.860000
max       336.478500
Name: Price_eur, dtype: float64

In [67]:
df.head(2)

,Scrape Date,Flight Number,Origin City,Origin Airport,Destination City,Destination Airport,Departure Date,Departure Time,Arrival Time,Price,Price_raw,currency,Price_clean,Price_eur
0,2026-02-18,FR 2663,Fráncfort Hahn,HHN,Alicante,ALC,2026-02-20,06:20,09:00,"156,99 €","156,99 €",€,156.99,156.99
1,2026-02-18,FR 2663,Fráncfort Hahn,HHN,Alicante,ALC,2026-02-21,17:45,20:25,"60,99 €","60,99 €",€,60.99,60.99


In [68]:
df[(df["currency"]=="£")]

,Scrape Date,Flight Number,Origin City,Origin Airport,Destination City,Destination Airport,Departure Date,Departure Time,Arrival Time,Price,Price_raw,currency,Price_clean,Price_eur
253,2026-02-18 16:18:28,FR 6199,Londres Stansted,STN,Fráncfort Hahn,HHN,2026-03-20,06:15,08:40,"15,70 £","15,70 £",£,15.70,18.0550
254,2026-02-18 16:18:28,FR 1749,Londres Stansted,STN,Fráncfort Hahn,HHN,2026-03-20,17:05,19:30,"14,99 £","14,99 £",£,14.99,17.2385
255,2026-02-18 16:18:28,FR 1749,Londres Stansted,STN,Fráncfort Hahn,HHN,2026-03-21,08:10,10:35,"14,99 £","14,99 £",£,14.99,17.2385
256,2026-02-18 16:18:28,FR 6199,Londres Stansted,STN,Fráncfort Hahn,HHN,2026-03-21,13:40,16:05,"14,99 £","14,99 £",£,14.99,17.2385
257,2026-02-18 16:18:28,FR 6207,Londres Stansted,STN,Fráncfort Hahn,HHN,2026-03-21,20:25,22:50,"17,99 £","17,99 £",£,17.99,20.6885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8857,2026-02-24 08:46:40,FR 2814,Londres Stansted,STN,Colonia,CGN,2026-04-17,14:30,16:50,"53,84 £","53,84 £",£,53.84,61.9160
8858,2026-02-24 08:46:40,FR 2816,Londres Stansted,STN,Colonia,CGN,2026-04-17,17:00,19:20,"20,99 £","20,99 £",£,20.99,24.1385
8859,2026-02-24 08:46:40,FR 2814,Londres Stansted,STN,Colonia,CGN,2026-04-18,09:00,11:20,"22,99 £","22,99 £",£,22.99,26.4385
8860,2026-02-24 08:46:40,RK 2816,Londres Stansted,STN,Colonia,CGN,2026-04-18,15:25,17:45,"14,99 £","14,99 £",£,14.99,17.2385


In [69]:
df["Price_eur"].isna().sum()

np.int64(0)

In [71]:
df.shape

(9114, 14)

## 7. Type conversion and data cleaning

After repairing the raw dataset structure, the next step is to convert the variables into appropriate data types.

At this stage, all columns are still stored as strings, which limits the possibility of performing numerical operations, date manipulations, or statistical analysis. Therefore, I convert the date variables into datetime format and clean the price variable so that it can be interpreted as numeric.

This step is essential for transforming the scraped dataset into an analysis-ready table.

## 10. Final data type conversion

After cleaning the raw price variable and converting all values into euros, the dataset is now ready for final type conversion.

At this stage, I convert the date variables into datetime format and ensure that the standardized price variable (`Price_eur`) is stored as a numeric value. This step prepares the dataset for analysis and modeling.

In [72]:
# Convert date columns to datetime format
df["Scrape Date"] = pd.to_datetime(df["Scrape Date"], errors="coerce")
df["Departure Date"] = pd.to_datetime(df["Departure Date"], errors="coerce")

# Ensure price in euros is numeric
df["Price_eur"] = pd.to_numeric(df["Price_eur"], errors="coerce")

# Optional: drop any remaining missing prices
df = df.dropna(subset=["Price_eur"])

# Reset index after cleaning
df = df.reset_index(drop=True)

# Display updated data types
df.dtypes

Scrape Date            datetime64[us]
Flight Number                     str
Origin City                       str
Origin Airport                    str
Destination City                  str
Destination Airport               str
Departure Date         datetime64[us]
Departure Time                    str
Arrival Time                      str
Price                             str
Price_raw                         str
currency                          str
Price_clean                   float64
Price_eur                     float64
dtype: object

In [73]:
# Convert time columns to datetime format
df["Departure Time"] = pd.to_datetime(df["Departure Time"], format="%H:%M", errors="coerce")
df["Arrival Time"] = pd.to_datetime(df["Arrival Time"], format="%H:%M", errors="coerce")

# Calculate flight duration
df["flight_duration"] = df["Arrival Time"] - df["Departure Time"]

df[["Departure Time", "Arrival Time", "flight_duration"]].head()

,Departure Time,Arrival Time,flight_duration
0,1900-01-01 06:20:00,1900-01-01 09:00:00,0 days 02:40:00
1,1900-01-01 17:45:00,1900-01-01 20:25:00,0 days 02:40:00
2,1900-01-01 13:15:00,1900-01-01 15:55:00,0 days 02:40:00
3,1900-01-01 19:50:00,1900-01-01 22:30:00,0 days 02:40:00
4,1900-01-01 15:10:00,1900-01-01 17:50:00,0 days 02:40:00


In [74]:
# Fix overnight flights (arrival next day)
df.loc[df["flight_duration"] < pd.Timedelta(0), "flight_duration"] += pd.Timedelta(days=1)

In [75]:
df["flight_duration"].describe()

count                      9114
mean     0 days 02:10:49.907834
std      0 days 00:47:48.177419
min             0 days 00:20:00
25%             0 days 02:05:00
50%             0 days 02:20:00
75%             0 days 02:30:00
max             0 days 05:40:00
Name: flight_duration, dtype: object

In [ ]:
df["duration_minutes"] = df["flight_duration"].dt.total_seconds() / 60 # Convert duration to minutes

In [78]:
df.head(2)

,Scrape Date,Flight Number,Origin City,Origin Airport,Destination City,Destination Airport,Departure Date,Departure Time,Arrival Time,Price,Price_raw,currency,Price_clean,Price_eur,flight_duration,duration_minutes
0,2026-02-18,FR 2663,Fráncfort Hahn,HHN,Alicante,ALC,2026-02-20,1900-01-01 06:20:00,1900-01-01 09:00:00,"156,99 €","156,99 €",€,156.99,156.99,0 days 02:40:00,160.0
1,2026-02-18,FR 2663,Fráncfort Hahn,HHN,Alicante,ALC,2026-02-21,1900-01-01 17:45:00,1900-01-01 20:25:00,"60,99 €","60,99 €",€,60.99,60.99,0 days 02:40:00,160.0
